In [2]:
import pandas as pd 
import sys
import os
import numpy as np
name = '100K_accts_MID5'


In [ ]:
path_tx = '../flib/AMLsim/outputs/'+name+'/tx_log.csv'
path_node = '../flib/AMLsim/outputs/'+name+'/node_log.csv'

In [4]:
# GNN Preprocessing
path_tx = '~/flib/gnn/data/'+name+'/bank/train/edges.csv'
path_node = '~/flib/gnn/data/'+name+'/bank/train/nodes.csv'

In [5]:
data = pd.read_csv(path_tx)
nodes = pd.read_csv(path_node)

In [6]:
data.head()

,src,dst,sum,mean,median,std,max,min,count,is_sar
0,0,55445,4485.25,498.361111,262.85,396.608623,1120.69,174.77,9,0
1,0,88454,6385.17,638.517000,408.94,492.467352,1487.10,83.81,10,0
2,1,46115,5703.94,950.656667,917.73,356.195264,1394.38,611.34,6,0
3,1,97518,3509.62,438.702500,376.22,288.054803,844.00,30.01,8,0
4,2,87852,2694.78,449.130000,433.87,333.169060,827.27,83.89,6,0


In [6]:
# If bankOrig = source and isSAR = 1, keep those rows
sars = data[(data['bankDest'] == 'sink') & (data['isSAR'] == 1)]

In [7]:
print(sars.head())

        step      type  amount  nameOrig bankOrig  daysInBankOrig  \
100001     0  TRANSFER  644.14     53663     bank               0   
100013     0  TRANSFER  253.64     50535     bank               0   
100038     0  TRANSFER  124.90     20248     bank               0   
100065     0  TRANSFER  610.61     13702     bank               0   
100102     0  TRANSFER  439.28     62653     bank               0   

        phoneChangesOrig  oldbalanceOrig  newbalanceOrig  nameDest bankDest  \
100001                 0       139060.90       138416.76        -1     sink   
100013                 0       113697.11       113443.47        -1     sink   
100038                 0       188554.84       188429.93        -1     sink   
100065                 0        81531.11        80920.50        -1     sink   
100102                 0       175525.21       175085.93        -1     sink   

        daysInBankDest  phoneChangesDest  oldbalanceDest  newbalanceDest  \
100001               0            

In [9]:
data = data[data['type'] != 'INITALBALANCE']

In [8]:
print(len(data))

100000


In [9]:
# If type coumn = INITIALBALANCE remove the rows
source = data[data['bankOrig'] == 'source']
sink = data[data['bankDest'] == 'sink']
other = data[(data['bankOrig'] != 'source') & (data['bankDest'] != 'sink')]

print('Sink:',len(sink))
print('Source',len(source))
print('Internal',len(other))

KeyError: 'bankOrig'

In [34]:
# If type coumn = INITIALBALANCE remove the rows
data = data[data['type'] != 'INITALBALANCE']
data = data[data['bankOrig'] != 'source']
data = data[data['bankDest'] != 'sink']

In [35]:
# print len of data
print('Length of data: ', len(data))

Length of data:  784549


In [ ]:
# MID5
# 25231016
#   159766 UTAN SINK O SOURCE

# MID5 TESTING
# 22093757
#   784549 UTAN SINK O SOURCE

In [12]:
# Nodes
SAR = nodes[nodes['is_sar'] == 1]
NORMAL = nodes[nodes['is_sar'] == 0]

# Total amout of SAR rows
print('Total SAR:', SAR.shape[0])
print('Total NORMAL:', NORMAL.shape[0])

sar_frac = SAR.shape[0]/(SAR.shape[0]+NORMAL.shape[0])
print('Fraction of SAR:', sar_frac)

normal_frac = NORMAL.shape[0]/(SAR.shape[0]+NORMAL.shape[0])
print('Fraction of normal:', normal_frac)

Total SAR: 4678
Total NORMAL: 95322
Fraction of SAR: 0.04678
Fraction of normal: 0.95322


In [12]:
# Transactions
SAR = data[data['isSAR'] == 1]
NORMAL = data[data['isSAR'] == 0]

# Total amout of SAR rows
print('Total SAR:', SAR.shape[0])
print('Total NORMAL:', NORMAL.shape[0])
 
sar_frac = SAR.shape[0]/(SAR.shape[0]+NORMAL.shape[0])
print('Fraction of SAR:', sar_frac)

normal_frac = NORMAL.shape[0]/(SAR.shape[0]+NORMAL.shape[0])
print('Fraction of normal:', normal_frac)

Total SAR: 6160385
Total NORMAL: 20173443
Fraction of SAR: 0.23393427647511025
Fraction of normal: 0.7660657235248898


In [31]:
sys.path.insert(1,'../StatModels')
from tools import proccess_data

features, labels, _ = proccess_data(data,nodes)

In [32]:
# Join labels and features on "node_id"

df = pd.merge(features, labels, on='node_id')

SAR = df[df['IS_SAR']==1]
NORMAL = df[df['IS_SAR']==0]

print(SAR.head())

    node_id  avg_amount_sent  total_amount_sent  total_transactions_sent  \
36       36          1223.31          252001.41                      206   
56       56          1759.21          114348.70                       65   
70       70           818.79          156389.25                      191   
77       77           619.16          135595.27                      219   
99       99          1403.64          283535.34                      202   

    avg_amount_rec  total_amount_rec  total_transactions_rec  \
36         7156.27         336344.67                      47   
56         4598.21         243705.07                      53   
70         3101.49         145769.93                      47   
77         1991.86          93617.33                      47   
99        12554.93         477087.51                      38   

    avg_phone_changes  total_phone_changes  INIT_BALANCE  BANK_ID  \
36                0.0                    0      32989.96        0   
56                0.

In [33]:
# Calculate average of each feature for SAR and NORMAL

SAR_avg = SAR.mean()
NORMAL_avg = NORMAL.mean()

print("Sar")
print(SAR_avg)


Sar
node_id                     49796.663169
avg_amount_sent              1399.110186
total_amount_sent          232567.752332
total_transactions_sent       185.916503
avg_amount_rec               7864.490219
total_amount_rec           403181.839742
total_transactions_rec         52.126339
avg_phone_changes               0.002745
total_phone_changes             0.556066
INIT_BALANCE                51161.612990
BANK_ID                         0.000000
unique_accounts_sent            2.586448
unique_accounts_rec             2.593443
IS_SAR                          1.000000
dtype: float64


In [34]:
print("Normal")
print(NORMAL_avg)

Normal
node_id                     50009.224690
avg_amount_sent              1337.641455
total_amount_sent          271491.931830
total_transactions_sent       201.377679
avg_amount_rec               7516.095084
total_amount_rec           377886.991925
total_transactions_rec         51.235001
avg_phone_changes               0.000618
total_phone_changes             0.124800
INIT_BALANCE                50539.265633
BANK_ID                         0.000000
unique_accounts_sent            1.559151
unique_accounts_rec             1.558816
IS_SAR                          0.000000
dtype: float64


In [35]:
print((SAR_avg-NORMAL_avg)/NORMAL_avg)

node_id                   -0.004250
avg_amount_sent            0.045953
total_amount_sent         -0.143371
total_transactions_sent   -0.076777
avg_amount_rec             0.046353
total_amount_rec           0.066938
total_transactions_rec     0.017397
avg_phone_changes          3.441769
total_phone_changes        3.455669
INIT_BALANCE               0.012314
BANK_ID                         NaN
unique_accounts_sent       0.658882
unique_accounts_rec        0.663726
IS_SAR                          inf
dtype: float64


In [36]:
# node_id                     -0.004250
# avg_amount_sent              0.039119
# total_amount_sent           -0.155761
# total_transactions_sent     -0.086223
# avg_amount_rec              -0.061749
# total_amount_rec             0.001447
# total_transactions_rec       0.061659
# avg_phone_changes          180.246818
# total_phone_changes        172.111937
# INIT_BALANCE                 0.012314
# BANK_ID                           NaN
# unique_accounts_sent         0.648928
# unique_accounts_rec          0.654987
# IS_SAR                            inf
# dtype: float64